In [27]:
options(repos = c(CRAN = "https://cloud.r-project.org"))

if (!require("pacman")) install.packages("pacman")
pacman::p_load(tidyverse, ggplot2, dplyr, lubridate)
library(readr)
library(magrittr)

# Read v2010 data and compute uncomp_care

In [28]:
# Load the HCRIS Data for 2010
final.hcris.v2010 <- read_csv("../data/output/HCRIS_Data_v2010.csv")
 
# Check column names
print(colnames(final.hcris.v2010))
 
# Compute uncomp_care from S-10 components if the column exists
if ("tot_uncomp_care_partial_pmts" %in% colnames(final.hcris.v2010)) {
  final.hcris <- final.hcris.v2010 %>%
    mutate(
      uncomp_care = tot_uncomp_care_charges - tot_uncomp_care_partial_pmts + bad_debt,
      fy_end = mdy(fy_end),  # Correctly parse the date
      fy_start = mdy(fy_start),  # Correctly parse the date
      date_processed = mdy(date_processed),  # Correctly parse the date
      date_created = mdy(date_created),  # Correctly parse the date
      tot_discounts = abs(tot_discounts),
      herp_payment = abs(hrrp_payment),
      fyear = year(fy_end)
    ) %>%
    arrange(provider_number, fyear) %>%
    select(-year)
 
  # Count of hospitals/provider_number by year
  final.hcris %>%
    group_by(fyear) %>%
    count()
} else {
  warning("Column 'tot_uncomp_care_partial_pmts' not found in the dataset.")
}

Rows: 57802 Columns: 48
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (12): provider_number, fy_start, fy_end, date_processed, date_created, d...
dbl (35): report, status, year, beds, tot_charges, tot_discounts, net_pat_re...
lgl  (1): npi

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


 [1] "report"                       "provider_number"             
 [3] "npi"                          "fy_start"                    
 [5] "fy_end"                       "date_processed"              
 [7] "date_created"                 "status"                      
 [9] "year"                         "data_source"                 
[11] "beds"                         "tot_charges"                 
[13] "tot_discounts"                "net_pat_rev"                 
[15] "tot_operating_exp"            "ip_charges"                  
[17] "icu_charges"                  "ancillary_charges"           
[19] "tot_discharges"               "mcare_discharges"            
[21] "mcaid_discharges"             "tot_mcare_payment"           
[23] "secondary_mcare_payment"      "street"                      
[25] "city"                         "state"                       
[27] "zip"                          "county"                      
[29] "name"                         "hvbp_payment"            

fyear,n
<dbl>,<int>
2011,5860
2012,6231
2013,6159
2014,6168
2015,6154
2016,6188
2017,6176
2018,6158
2019,6134


# Clean duplicate reports

In [29]:
# Count of reports by hospital fiscal year
final.hcris <- final.hcris %>%
  add_count(provider_number, fyear, name = "total_reports")
 
# Create running total of reports
final.hcris <- final.hcris %>%
  group_by(provider_number, fyear) %>%
  mutate(report_number = row_number())
 
# Identify hospitals with only one report per fiscal year
unique.hcris <- final.hcris %>%
  filter(total_reports == 1) %>%  # Use == for comparison
  select(-report, -total_reports, -report_number, -npi, -status) %>%
  mutate(source = 'unique reports')
 
# View the output
print(unique.hcris)

# A tibble: 56,130 × 48
# Groups:   provider_number, fyear [56,130]
   provider_number fy_start   fy_end     date_processed date_created data_source
   <chr>           <date>     <date>     <date>         <date>       <chr>      
 1 010001          2010-10-01 2011-09-30 2012-11-09     2012-11-09   v2010      
 2 010001          2011-10-01 2012-09-30 2013-11-15     2013-11-15   v2010      
 3 010001          2012-10-01 2013-09-30 2014-09-26     2014-03-17   v2010      
 4 010001          2013-10-01 2014-09-30 2015-03-13     2015-03-11   v2010      
 5 010001          2014-10-01 2015-09-30 2016-04-28     2016-04-18   v2010      
 6 010001          2015-10-01 2016-09-30 2019-03-20     2019-03-17   v2010      
 7 010001          2016-10-01 2017-09-30 2018-07-19     2018-07-10   v2010      
 8 010001          2017-10-01 2018-09-30 2021-01-19     2021-01-18   v2010      
 9 010001          2018-10-01 2019-09-30 2021-12-28     2021-12-27   v2010      
10 010005          2010-10-01 2011-09-30 

In [30]:
# Identify hospitals with multiple reports per fiscal year
duplicate.hcris <- final.hcris %>%
  filter(total_reports > 1) %>%
  mutate(time_diff = fy_end - fy_start)  # Calculate the elapsed time
 
# Calculate the total elapsed time
duplicate.hcris <- duplicate.hcris %>%
  group_by(provider_number, fyear) %>%
  mutate(total_days = sum(time_diff))
 
# Aggregate data if elapsed time sums to around 365 days
unique_hcris2 <- duplicate.hcris %>%
  filter(total_days < 370) %>%
  group_by(provider_number, fyear) %>%
  mutate(
    hrrp_payment = if_else(is.na(hrrp_payment), 0, hrrp_payment),
    hvbp_payment = if_else(is.na(hvbp_payment), 0, hvbp_payment)
  ) %>%
  summarize(
    beds = max(beds),
    tot_charges = sum(tot_charges),
    tot_discounts = sum(tot_discounts),
    tot_operating_exp = sum(tot_operating_exp),
    ip_charges = sum(ip_charges),
    icu_charges = sum(icu_charges),
    ancillary_charges = sum(ancillary_charges),
    tot_discharges = sum(tot_discharges),
    mcare_discharges = sum(mcare_discharges),
    mcaid_discharges = sum(mcaid_discharges),
    tot_care_payment = sum(tot_mcare_payment),
    secondary_mcare_payment = sum(secondary_mcare_payment),
    hvbp_payment = sum(hvbp_payment),
    hrrp_payment = sum(hrrp_payment),
    fy_start = min(fy_start),
    fy_end = max(fy_end),
    date_processed = max(date_processed),
    date_created = min(date_created),
    street = first(street),
    city = first(city),
    state = first(state),
    zip = first(zip),
    county = first(county),
    uncomp_care = sum(uncomp_care, na.rm = TRUE),
    tot_uncomp_care_charges = sum(tot_uncomp_care_charges, na.rm = TRUE),
    tot_uncomp_care_partial_pats = sum(tot_uncomp_care_partial_pmts, na.rm = TRUE),
    bad_debt = sum(bad_debt, na.rm = TRUE),
    cost_to_charge = first(cost_to_charge)
  ) %>%
  mutate(source = 'total for year')
 
# Elapsed time exceeding 370 days
duplicate.hcris2 <- duplicate.hcris %>%
  filter(total_days > 370) %>%
  mutate(max_days = max(time_diff), max_date = max(fy_end))
 
# View the outputs
print(unique_hcris2)
print(duplicate.hcris2)

`summarise()` has regrouped the output.
ℹ Summaries were computed grouped by provider_number and fyear.
ℹ Output is grouped by provider_number.
ℹ Use `summarise(.groups = "drop_last")` to silence this message.
ℹ Use `summarise(.by = c(provider_number, fyear))` for per-operation grouping
  (`?dplyr::dplyr_by`) instead.


# A tibble: 231 × 31
# Groups:   provider_number [224]
   provider_number fyear  beds tot_charges tot_discounts tot_operating_exp
   <chr>           <dbl> <dbl>       <dbl>         <dbl>             <dbl>
 1 010016           2015   212   929125972     767378931         151777722
 2 010089           2015   145   418840116     325779425          91578517
 3 010101           2015    81   150441054     118162658          33000798
 4 010103           2015   273  1380522836    1156099251         190122995
 5 030074           2016    11          NA            NA         147521930
 6 030077           2015     8          NA            NA          35575204
 7 030137           2018    49   103954086      80182272          57756067
 8 032004           2015    32    26518821      19272480           8883415
 9 040018           2018    39   173592475     154746277          18891612
10 040074           2012    30    77538088      51783554          29953728
# ℹ 221 more rows
# ℹ 25 more variables: ip_c

In [31]:
# Identify primary reports covering the full year
unique_hcris3 <- duplicate.hcris2 %>%
  filter(max_days == time_diff, time_diff > 360, max_date == fy_end) %>%
  select(-report, -total_reports, -report_number, -npi, -status, -max_days, -time_diff, -total_days, -max_date) %>%
  mutate(source = 'primary report')
 
# Remaining hospitals - anti join to get those not in unique_hcris3
duplicate_hcris3 <- anti_join(duplicate.hcris2, unique_hcris3, by = c("provider_number", "fyear"))
 
# Ensure total_days and time_diff are integers
duplicate_hcris3 <- duplicate_hcris3 %>%
  mutate(
    total_days = as.integer(total_days),
    time_diff = as.integer(time_diff)
  ) %>%
  mutate_at(
    vars(tot_charges, tot_discounts, tot_operating_exp, ip_charges,
         icu_charges, ancillary_charges, hrrp_payment, hvbp_payment, 
         uncomp_care, tot_uncomp_care_charges, tot_uncomp_care_partial_pmts, 
         bad_debt),
    list(~ . * (time_diff / total_days))
  )
 
# View the outputs
print(unique_hcris3)
print(duplicate_hcris3)

# A tibble: 8 × 48
# Groups:   provider_number, fyear [8]
  provider_number fy_start   fy_end     date_processed date_created data_source
  <chr>           <date>     <date>     <date>         <date>       <chr>      
1 010139          2015-01-01 2015-12-31 2016-06-15     2016-06-13   v2010      
2 050393          2013-10-01 2014-09-30 2015-12-08     2015-12-07   v2010      
3 110190          2013-01-01 2013-12-31 2014-06-25     2014-06-23   v2010      
4 180021          2017-07-01 2018-06-30 2018-12-27     2018-12-18   v2010      
5 194007          2013-01-02 2013-12-31 2014-08-25     2014-08-21   v2010      
6 230066          2013-07-01 2014-06-30 2014-12-24     2014-12-23   v2010      
7 370078          2013-07-01 2014-06-30 2014-12-11     2014-12-10   v2010      
8 451370          2013-01-01 2013-12-31 2014-09-25     2014-09-24   v2010      
# ℹ 42 more variables: beds <dbl>, tot_charges <dbl>, tot_discounts <dbl>,
#   net_pat_rev <dbl>, tot_operating_exp <dbl>, ip_charges <dbl>,
#

In [32]:
unique_hcris4 <- duplicate_hcris3 %>%
  group_by(provider_number, fyear) %>%
  filter(any(!is.na(beds))) %>%  # Filter out groups with all NA values
  mutate(
    hrrp_payment = if_else(is.na(hrrp_payment), 0, hrrp_payment),
    hvbp_payment = if_else(is.na(hvbp_payment), 0, hvbp_payment)
  ) %>%
  summarize(
    beds = max(beds, na.rm = TRUE),
    tot_charges = sum(tot_charges, na.rm = TRUE),
    tot_discounts = sum(tot_discounts, na.rm = TRUE),
    tot_operating_exp = sum(tot_operating_exp, na.rm = TRUE),
    ip_charges = sum(ip_charges, na.rm = TRUE),
    icu_charges = sum(icu_charges, na.rm = TRUE),
    ancillary_charges = sum(ancillary_charges, na.rm = TRUE),
    tot_discharges = sum(tot_discharges, na.rm = TRUE),
    mcare_discharges = sum(mcare_discharges, na.rm = TRUE),
    mcaid_discharges = sum(mcaid_discharges, na.rm = TRUE),
    tot_care_payment = sum(tot_mcare_payment, na.rm = TRUE),
    secondary_mcare_payment = sum(secondary_mcare_payment, na.rm = TRUE),
    hvbp_payment = sum(hvbp_payment, na.rm = TRUE),
    hrrp_payment = sum(hrrp_payment, na.rm = TRUE),
    fy_start = min(fy_start, na.rm = TRUE),
    fy_end = max(fy_end, na.rm = TRUE),
    date_processed = max(date_processed, na.rm = TRUE),
    date_created = min(date_created, na.rm = TRUE),
    street = first(street),
    city = first(city),
    state = first(state),
    zip = first(zip),
    county = first(county),
    uncomp_care = sum(uncomp_care, na.rm = TRUE),
    tot_uncomp_care_charges = sum(tot_uncomp_care_charges, na.rm = TRUE),
    tot_uncomp_care_partial_pmts = sum(tot_uncomp_care_partial_pmts, na.rm = TRUE),
    bad_debt = sum(bad_debt, na.rm = TRUE),
    cost_to_charge = first(cost_to_charge)
  ) %>%
  mutate(source = 'weighted average', .groups = "drop")  # Drop grouping
 
# View the output
print(unique_hcris4)

`summarise()` has regrouped the output.
ℹ Summaries were computed grouped by provider_number and fyear.
ℹ Output is grouped by provider_number.
ℹ Use `summarise(.groups = "drop_last")` to silence this message.
ℹ Use `summarise(.by = c(provider_number, fyear))` for per-operation grouping
  (`?dplyr::dplyr_by`) instead.


# A tibble: 584 × 32
# Groups:   provider_number [569]
   provider_number fyear  beds tot_charges tot_discounts tot_operating_exp
   <chr>           <dbl> <dbl>       <dbl>         <dbl>             <dbl>
 1 010018           2012    12   72203507.     52722318.         24897842.
 2 010019           2014   185  317909935.    259258801          69371516 
 3 010022           2018    45   77336180.     65007991.         14946171.
 4 010032           2017    34   18222585.     14071822.          7023164.
 5 010038           2017   125  344555341.    316067599.         29476635.
 6 010046           2015   256  760393609.    683837396.         75778142.
 7 010054           2011   109  240108319.    203712711.         41313441.
 8 010055           2014   235  993595674.    828984910.        173206568.
 9 010069           2017    30   57159865.     40365359.         16992728.
10 010078           2017   250          0             0         155866673 
# ℹ 574 more rows
# ℹ 26 more variables: ip_c

In [33]:
# Combine datasets
final_hcris_data <- bind_rows(unique_hcris, unique_hcris2, unique_hcris3, unique_hcris4)
 
# Rename 'fyear' to 'year'
final_hcris_data <- final_hcris_data %>%
  rename(year = fyear) %>%
  arrange(provider_number, year)  # Arrange by provider_number and year
 
# Write CSV files for each unique year
for (y in sort(unique(final_hcris_data$year))) {
  final_hcris_data %>%
    filter(year == y) %>%
    write_csv(paste('../data/output/data-', y, '.csv', sep = ''))  # Correct CSV file naming
}
 
# Print total rows written
cat("Years written:", sort(unique(final_hcris_data$year)), "\n")
cat("Total rows:", nrow(final_hcris_data), "\n")

ERROR: Error in eval(expr, envir, enclos): object 'unique_hcris' not found
